# NB1 · Regressione e bias-varianza, resi visibili

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/battles5/conad-statistical-learning/blob/main/notebooks/NB1_regressione_biasvarianza.ipynb)

**fit lineare e polinomiale su Auto, la manopola del grado, la U del test error**

corso *Apprendimento statistico* per CONAD Nord Ovest · Orso Peruzzi (IFAB)

> come si esegue una cella: clicca dentro e premi **Shift + Invio**. esegui le celle in ordine, dall'alto verso il basso.

## 1. Dati e setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

auto = pd.read_csv("https://raw.githubusercontent.com/battles5/conad-statistical-learning/main/data/Auto.csv")
auto["horsepower"] = pd.to_numeric(auto["horsepower"], errors="coerce")
auto = auto.dropna(subset=["horsepower", "mpg"]).reset_index(drop=True)

X = auto[["horsepower"]].values
y = auto["mpg"].values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.30, random_state=0)
print("training:", len(Xtr), " test:", len(Xte))

## 2. La baseline lineare

una retta: `mpg = b0 + b1 * horsepower`. semplice e interpretabile.

In [ ]:
from sklearn.linear_model import LinearRegression

lin = LinearRegression().fit(Xtr, ytr)
mse_tr = mean_squared_error(ytr, lin.predict(Xtr))
mse_te = mean_squared_error(yte, lin.predict(Xte))
print(f"MSE training: {mse_tr:.1f}")
print(f"MSE test:     {mse_te:.1f}")

griglia = np.linspace(X.min(), X.max(), 200).reshape(-1, 1)
plt.figure(figsize=(7, 5))
plt.scatter(Xtr, ytr, alpha=0.4, color="#00ADCF", label="training")
plt.plot(griglia, lin.predict(griglia), color="#002060", lw=2.5, label="retta")
plt.xlabel("horsepower"); plt.ylabel("mpg"); plt.legend(); plt.title("Fit lineare")
plt.show()

la retta non coglie la curvatura dei dati: e' un caso di **distorsione (bias) alta**, il modello e' troppo rigido.

## 3. Saliamo in flessibilita': i polinomi

aggiungiamo `horsepower^2`, `horsepower^3`, ... la manopola e' il **grado** del polinomio.

> **manopola**: cambia `GRADO` (prova 1, 2, 5, 10, 15) e osserva come cambia la curva e i due MSE.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# >>> MANOPOLA: grado del polinomio <<<
GRADO = 2

modello = make_pipeline(PolynomialFeatures(GRADO), LinearRegression()).fit(Xtr, ytr)
mse_tr = mean_squared_error(ytr, modello.predict(Xtr))
mse_te = mean_squared_error(yte, modello.predict(Xte))
print(f"grado {GRADO}  ->  MSE training: {mse_tr:.1f}   MSE test: {mse_te:.1f}")

plt.figure(figsize=(7, 5))
plt.scatter(Xtr, ytr, alpha=0.4, color="#00ADCF", label="training")
plt.plot(griglia, modello.predict(griglia), color="#DC4C4C", lw=2.5, label=f"polinomio grado {GRADO}")
plt.xlabel("horsepower"); plt.ylabel("mpg"); plt.legend(); plt.title(f"Fit polinomiale (grado {GRADO})")
plt.ylim(5, 50)
plt.show()

> **indovina prima di eseguire**: cosa succede al **MSE di training** e al **MSE di test** se metti `GRADO = 15`? quale dei due peggiora?

provalo, poi continua.

## 4. La U del test error

invece di provare i gradi a mano, li proviamo tutti in un ciclo e disegniamo le due curve.

In [ ]:
gradi = range(1, 16)
err_tr, err_te = [], []
for g in gradi:
    m = make_pipeline(PolynomialFeatures(g), LinearRegression()).fit(Xtr, ytr)
    err_tr.append(mean_squared_error(ytr, m.predict(Xtr)))
    err_te.append(mean_squared_error(yte, m.predict(Xte)))

plt.figure(figsize=(7.5, 5))
plt.plot(list(gradi), err_tr, "o-", color="#9aa3b2", label="MSE training")
plt.plot(list(gradi), err_te, "o-", color="#DC4C4C", label="MSE test")
plt.xlabel("grado del polinomio (flessibilità)"); plt.ylabel("MSE")
plt.title("Training error che scende, test error a U")
plt.ylim(15, 40); plt.legend(); plt.show()

migliore = list(gradi)[int(np.argmin(err_te))]
print("grado con MSE di test minimo:", migliore)

ecco il trade-off bias-varianza con i tuoi occhi: il training error scende sempre, ma il test error prima cala e poi risale. il modello migliore sta nel minimo della U, non al grado piu' alto.

---

### Cella bonus: la cross-validation

invece di un solo split, la k-fold cross-validation media l'errore su piu' divisioni: una stima piu' stabile del test error.

In [ ]:
# BONUS
from sklearn.model_selection import cross_val_score

cv_mse = []
for g in gradi:
    m = make_pipeline(PolynomialFeatures(g), LinearRegression())
    punteggi = cross_val_score(m, X, y, cv=10, scoring="neg_mean_squared_error")
    cv_mse.append(-punteggi.mean())

plt.figure(figsize=(7.5, 4.5))
plt.plot(list(gradi), cv_mse, "o-", color="#002060")
plt.xlabel("grado"); plt.ylabel("MSE in 10-fold CV"); plt.title("Cross-validation: scelta del grado")
plt.ylim(15, 40); plt.show()
print("grado scelto dalla CV:", list(gradi)[int(np.argmin(cv_mse))])